In [7]:
# Cell 0 - Install Chronos dependencies
!pip install chronos-forecasting transformers accelerate --quiet


ERROR: Will not install to the user site because it will lack sys.path precedence to transformers in /opt/jupyterhub/jupyter_env/lib/python3.12/site-packages


In [9]:
import sys
!{sys.executable} -m pip install chronos-forecasting --target=/tmp/chronos_pkg
sys.path.insert(0, "/tmp/chronos_pkg")

  Using cached chronos_forecasting-2.2.2-py3-none-any.whl.metadata (23 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached transformers-4.57.6-py3-none-any.whl.metadata (43 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.9 MB/s eta 0:00:00
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 19.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 33.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 M

# Chronos zero-shot baseline

This notebook is intentionally apples-to-oranges with the feature-aware model notebooks. Chronos is a pretrained time-series foundation model, so here it sees only the raw `HB_HUBAVG` target history and none of the engineered ERCOT, weather, gas, outage, load, lag, rolling, stress, or GDELT features.

Read this as a "what does a generic pretrained model think?" baseline, not as a direct competitor to the supervised feature-matrix models in `02_v2_models.ipynb`.

In [10]:
# Cell 1 - Setup: imports, paths
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import torch

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

from models.utils import (
    chronological_split,
    select_enhanced_features,
    TARGET_REG,
    regression_report,
    apply_feature_transforms,
)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print(f"torch: {torch.__version__}")
print(f"cuda available: {torch.cuda.is_available()}")


torch: 2.6.0+cu124
cuda available: True


In [11]:
# Cell 2 - Load feature matrix, chronological split, feature list
# Chronos will use only TARGET_REG values; loading the shared matrix keeps splits comparable.
FEATURES = PROJECT_ROOT / "data" / "features"

mat = pd.read_parquet(FEATURES / "feature_matrix_engineered_v2.parquet")
mat = apply_feature_transforms(mat)
train, val, test = chronological_split(mat)

features = select_enhanced_features(mat)
print(f"\n{len(features)} enhanced features selected for reference only")
print("Chronos input columns:", [TARGET_REG])


  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,429 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)

63 enhanced features selected for reference only
Chronos input columns: ['HB_HUBAVG']


In [12]:
# Cell 3 - Shared regression target split only
# No X matrices are built here: Chronos is zero-shot on the raw HB_HUBAVG series.
df_reg = mat.dropna(subset=[TARGET_REG]).sort_index()
tr_reg, vl_reg, te_reg = chronological_split(df_reg)

y_train_r = tr_reg[TARGET_REG]
y_val_r = vl_reg[TARGET_REG]
y_test_r = te_reg[TARGET_REG]

print(f"Regression target train/val/test: {y_train_r.shape}, {y_val_r.shape}, {y_test_r.shape}")
print(f"Test target window: {y_test_r.index.min()} -> {y_test_r.index.max()}")


  Split boundaries:
    Train: 2016-01-31 06:00:00+00:00 → 2022-12-31 23:00:00+00:00 (60,428 rows)
    Val:   2023-01-01 00:00:00+00:00 → 2023-12-31 23:00:00+00:00 (8,760 rows)
    Test:  2024-01-01 00:00:00+00:00 → 2024-12-31 06:00:00+00:00 (8,767 rows)
Regression target train/val/test: (60428,), (8760,), (8767,)
Test target window: 2024-01-01 00:00:00+00:00 -> 2024-12-31 06:00:00+00:00


In [13]:
# Cell 4 - A. Load pretrained Chronos pipeline
from chronos import ChronosPipeline

# Start small for runtime/memory. Swap to "amazon/chronos-t5-base" if quality is poor.
pipe = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/185M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

In [14]:
# Cell 5 - B. Rolling one-hour zero-shot forecasts over the 2024 test set
# For each test timestamp t, context is the prior 512 hours of raw HB_HUBAVG only.
# ChronosPipeline.predict accepts a list of 1D contexts for batched inference.

context_length = 512
prediction_length = 1
batch_size = 64  # Lower to 16/32 if GPU memory is tight; set to 1 for single-row inference.

target_series = df_reg[TARGET_REG].astype("float32").sort_index()
test_positions = target_series.index.get_indexer(y_test_r.index)

if (test_positions < 0).any():
    missing = y_test_r.index[test_positions < 0]
    raise ValueError(f"Could not align {len(missing)} test timestamps to target_series")

too_early = y_test_r.index[test_positions < context_length]
if len(too_early):
    raise ValueError(
        f"Need {context_length} prior hours for every test row; "
        f"first insufficient timestamp is {too_early[0]}"
    )

pred_values = []
pred_index = []

for start in tqdm(range(0, len(y_test_r), batch_size), desc="Chronos test forecasts"):
    batch_positions = test_positions[start:start + batch_size]
    batch_index = y_test_r.index[start:start + batch_size]
    contexts = [
        torch.tensor(
            target_series.iloc[pos - context_length:pos].to_numpy(),
            dtype=torch.float32,
        )
        for pos in batch_positions
    ]

    with torch.inference_mode():
        forecast_samples = pipe.predict(
            contexts,
            prediction_length=prediction_length,
        )

    if forecast_samples.ndim != 3:
        raise ValueError(f"Expected Chronos forecast shape (batch, samples, horizon), got {tuple(forecast_samples.shape)}")

    # Shape: (batch, sample_paths, 1). Score the median sample path for the one-hour horizon.
    batch_median = forecast_samples[:, :, 0].median(dim=1).values.detach().cpu().numpy()
    pred_values.extend(batch_median.tolist())
    pred_index.extend(batch_index)

chronos_pred = pd.Series(pred_values, index=pd.DatetimeIndex(pred_index), name="chronos_t5_small_median")
chronos_pred = chronos_pred.reindex(y_test_r.index)

print(chronos_pred.head())
print(f"\nPredictions: {chronos_pred.notna().sum():,} / {len(y_test_r):,}")


Chronos test forecasts:   0%|          | 0/137 [00:00<?, ?it/s]

datetime
2024-01-01 00:00:00+00:00    23.380308
2024-01-01 01:00:00+00:00    26.482870
2024-01-01 02:00:00+00:00    13.575462
2024-01-01 03:00:00+00:00    11.648534
2024-01-01 04:00:00+00:00    10.364736
Name: chronos_t5_small_median, dtype: float64

Predictions: 8,767 / 8,767


In [15]:
# Cell 6 - C. Assemble aligned predictions and report regression metrics
if chronos_pred.isna().any():
    raise ValueError(f"Chronos predictions contain {chronos_pred.isna().sum()} NaN values")

print("\nTest-set metrics:")
chronos_report = regression_report(
    y_test_r.values,
    chronos_pred.values,
    name="chronos_t5_small_zero_shot_raw_target",
)
chronos_report



Test-set metrics:

--- Regression report: chronos_t5_small_zero_shot_raw_target ---
  MAE:             $14.86/MWh
  RMSE:            $116.42/MWh
  Spike recall:    41.23%
  Spike precision: 59.49%
---------------------------------



{'name': 'chronos_t5_small_zero_shot_raw_target',
 'mae': 14.856474284457837,
 'rmse': np.float64(116.42123377156787),
 'spike_recall': np.float64(0.41228070175438597),
 'spike_precision': np.float64(0.5949367088607594)}

## Classifier comparison

Chronos is used here as a regression-only zero-shot forecaster over raw `HB_HUBAVG` values. It does not produce a calibrated `future_spike_24h` probability, so the spike-classifier comparison is N/A.

## Hybrid: Chronos forecast as a feature for XGBoost

Append-only section. The existing zero-shot Chronos results above stay untouched.
Here we feed Chronos's 1-hour forecast as a single new column into a feature-augmented
XGBoost that also sees the full 87-feature matrix.

```
Raw price series ──► Chronos (512h context) ──► chronos_pred
                                                    │
87 engineered features ────────────────────────────┐│
                                                   ▼▼
                                  XGBoost([87 features + chronos_pred] → price)
```

We need Chronos predictions on **train + val + test** so XGBoost has labels to learn
from. The cells above already produced `chronos_pred` for the 2024 test window; the
next cell generates predictions for the rest of the data using identical context/batch
settings, then concatenates.

In [ ]:
# Hybrid cell A — Chronos predictions for train + val (test already done above)
import xgboost as xgb

all_positions = np.arange(context_length, len(target_series))
test_pos_set = set(test_positions.tolist())
tv_positions = np.array([p for p in all_positions if p not in test_pos_set])

tv_pred_values = []
tv_pred_index = []
for start in tqdm(range(0, len(tv_positions), batch_size), desc="Chronos train+val forecasts"):
    batch_positions = tv_positions[start:start + batch_size]
    batch_index = target_series.index[batch_positions]
    contexts = [
        torch.tensor(
            target_series.iloc[p - context_length:p].to_numpy(),
            dtype=torch.float32,
        )
        for p in batch_positions
    ]
    with torch.inference_mode():
        forecast_samples = pipe.predict(contexts, prediction_length=prediction_length)
    batch_median = forecast_samples[:, :, 0].median(dim=1).values.detach().cpu().numpy()
    tv_pred_values.extend(batch_median.tolist())
    tv_pred_index.extend(batch_index)

chronos_tv_pred = pd.Series(tv_pred_values, index=pd.DatetimeIndex(tv_pred_index), name="chronos_pred")
chronos_full_pred = pd.concat([chronos_tv_pred, chronos_pred.rename("chronos_pred")]).sort_index()
print(f"Full Chronos predictions: {len(chronos_full_pred):,} "
      f"(expect {len(target_series) - context_length:,})")

In [ ]:
# Hybrid cell B — build augmented feature matrix and train XGBoost on price
augmented = mat.loc[chronos_full_pred.index].copy()
augmented["chronos_pred"] = chronos_full_pred.values
features_aug = features + ["chronos_pred"]

aug_train = augmented[augmented.index < "2023-01-01"]
aug_val   = augmented[(augmented.index >= "2023-01-01") & (augmented.index < "2024-01-01")]
aug_test  = augmented[augmented.index >= "2024-01-01"]

X_train_h = aug_train[features_aug].ffill().fillna(0); y_train_h = aug_train[TARGET_REG]
X_val_h   = aug_val[features_aug].ffill().fillna(0);   y_val_h   = aug_val[TARGET_REG]
X_test_h  = aug_test[features_aug].ffill().fillna(0);  y_test_h  = aug_test[TARGET_REG]

print(f"{len(features_aug)} augmented features  train={len(X_train_h):,}  val={len(X_val_h):,}  test={len(X_test_h):,}")

xgb_aug = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    objective="reg:squarederror", tree_method="hist",
    early_stopping_rounds=30, eval_metric="rmse",
    n_jobs=-1, random_state=42,
)
xgb_aug.fit(X_train_h, y_train_h, eval_set=[(X_val_h, y_val_h)], verbose=10)

In [ ]:
# Hybrid cell C — report on test and rank of chronos_pred in feature importance
y_pred_test_h = xgb_aug.predict(X_test_h)
r_hybrid = regression_report(y_test_h.values, y_pred_test_h, "chronos-xgb-hybrid-test")

imp = pd.Series(xgb_aug.feature_importances_, index=features_aug).sort_values(ascending=False)
print("\nTop 20 features by importance:")
for name, val in imp.head(20).items():
    print(f"  {name:40s} {val:.4f}")
chronos_rank = list(imp.index).index("chronos_pred") + 1
print(f"\nchronos_pred rank: {chronos_rank} / {len(features_aug)}")

In [ ]:
from sklearn.metrics import average_precision_score
import numpy as np

# Variant 1: Chronos zero-shot
_y_true_dollar = np.asarray(y_test_r).reshape(-1)
_y_pred_dollar = np.asarray(chronos_pred).reshape(-1)
_spike_pr_auc = average_precision_score(
    (_y_true_dollar > 200).astype(int),
    _y_pred_dollar,
)
print(f"\n=== Bake-off PR-AUC (regressor-as-score, threshold $200) ===")
print(f"  Spike PR-AUC [chronos-zero-shot]:   {_spike_pr_auc:.3f}")

# Variant 2: Chronos + XGBoost hybrid
_y_true_h_dollar = np.asarray(y_test_h).reshape(-1)
_y_pred_h_dollar = np.asarray(y_pred_test_h).reshape(-1)
_spike_pr_auc_h = average_precision_score(
    (_y_true_h_dollar > 200).astype(int),
    _y_pred_h_dollar,
)
print(f"  Spike PR-AUC [chronos-xgb-hybrid]:  {_spike_pr_auc_h:.3f}")

In [ ]:
# Final — Feature importance histogram (Chronos+XGBoost hybrid)
# Dedicated full-width view of how the hybrid XGBoost weights the 87 engineered
# features alongside chronos_pred (the Chronos foundation-model forecast).
import matplotlib.pyplot as plt

hyb_importance = pd.Series(xgb_aug.feature_importances_, index=features_aug).sort_values(ascending=False)
top20 = hyb_importance.head(20).sort_values()

colors = ["firebrick" if name == "chronos_pred" else "steelblue" for name in top20.index]

fig, ax = plt.subplots(figsize=(11, 8))
top20.plot.barh(ax=ax, color=colors, edgecolor="white")
ax.set_title("Chronos+XGBoost hybrid — top 20 feature importances\n(chronos_pred highlighted in red when present)", fontsize=13, pad=12)
ax.set_xlabel("Importance score")
ax.set_ylabel("")
for i, (name, val) in enumerate(zip(top20.index, top20.values)):
    ax.text(val, i, f"  {val:.3f}", va="center", fontsize=9, color="dimgray")
plt.tight_layout()
plt.show()

chronos_rank_full = list(hyb_importance.index).index("chronos_pred") + 1
print("\nTop 20 features by hybrid-XGBoost importance:")
for rank, (name, val) in enumerate(hyb_importance.head(20).items(), start=1):
    marker = "  <-- Chronos forecast" if name == "chronos_pred" else ""
    print(f"  {rank:2d}. {name:40s} {val:.4f}{marker}")
print(f"\nchronos_pred overall rank: {chronos_rank_full} / {len(features_aug)}")
